[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/05_computer_use/05_computer_use.ipynb)

# 05 · Computer Use：截图→动作

配套讲解：`05_讲解.html`。本 notebook 用 PIL 从零搭一个**两屏状态机 toy app**（LoginScreen → DashboardScreen），
让一个 GUI agent 在纯像素观察下完成任务「登录并打开 Settings 卡片」，并完整跑通 computer use 评测链路：

1. **环境**：`render(state) -> PIL.Image`（观察）+ `apply_action(state, action) -> state`（转移）——一个最小的可判分 GUI 环境；
2. **动作协议**：`{"action": "click", "x": …, "y": …}` / `{"action": "type", "text": …}`（JSON）；
3. **策略双路径**：`Qwen/Qwen2-VL-2B-Instruct`（真 VLM，看图输出动作）/ **mock 脚本化策略**（保证无 GPU 也能全流程跑通）；
4. **set-of-marks**：把"坐标回归"降级为"选择题"，对比两种协议的解析/命中成功率；
5. **episode 评测**：N=5 episodes 统计步级成功率与 episode 成功率，对照错误级联公式 $S(n)=p^n$。

> **算力说明** <span style="color:#c00">GPU 建议</span>：VLM 路径需 `pip install transformers accelerate qwen-vl-utils`，
> 权重下载约 4.4 GB，显存 **bf16 约 5 GB（默认，Colab T4 够用）**；显存特别紧张时可选 4-bit（约 2.5 GB，需额外装 `bitsandbytes`）。
> 没有 GPU 完全不影响：所有实验都有 mock 回退，纯 CPU 几秒跑完。

In [ ]:
import copy
import importlib.util
import json
import math
import random
import re

from PIL import Image, ImageDraw

HAS_TORCH = importlib.util.find_spec("torch") is not None
HAS_TRANSFORMERS = importlib.util.find_spec("transformers") is not None
HAS_QWEN_UTILS = importlib.util.find_spec("qwen_vl_utils") is not None
HAS_GPU = False
if HAS_TORCH:
    import torch
    HAS_GPU = torch.cuda.is_available()

USE_VLM = HAS_TORCH and HAS_TRANSFORMERS and HAS_QWEN_UTILS and HAS_GPU
# 想强制走 mock（不下载任何模型）：取消下一行注释
# USE_VLM = False
print(f"torch={HAS_TORCH} transformers={HAS_TRANSFORMERS} "
      f"qwen_vl_utils={HAS_QWEN_UTILS} gpu={HAS_GPU} -> USE_VLM={USE_VLM}")

## 1. Toy 环境：两屏状态机

GUI 环境本质是一个状态机：`state` 是唯一事实来源，截图只是它的**有损投影**（讲解第 1 节的 POMDP）。

```
 LoginScreen                                DashboardScreen
 ┌──────────────────────┐  login_button     ┌──────────────────────────┐
 │ [username_field]     │  (需 username 和   │ [Profile][Settings][Billing]│
 │ [password_field]     │   password 非空)   │                  [Log out] │
 │ [login_button]       │ ────────────────► │ 点 Settings 卡片 → 任务完成  │
 └──────────────────────┘ ◄──────────────── └──────────────────────────┘
                              logout_button
```

- `state = {"screen", "username", "password", "focus", "error", "opened_card"}`；
- 每个元素 = `{"id", "type", "bbox", "label"}`，**列表顺序即 z 序**（越靠后越上层）；
- `click` 做命中检测：input → 聚焦，button → 触发转移；`type` 写入当前聚焦的输入框；
- `task_success(state)`：**结果态检查**（讲解第 5 节）——只看终态，不看轨迹。

最后用 oracle（直接读 ground-truth state 的作弊策略，用来验证环境本身正确）演练一条标准轨迹：恰好 **6 步**。

In [ ]:
W, H = 640, 400


def login_elements(state):
    return [
        {"id": "username_field", "type": "input",  "bbox": (200, 120, 460, 155), "label": "Username"},
        {"id": "password_field", "type": "input",  "bbox": (200, 180, 460, 215), "label": "Password"},
        {"id": "login_button",   "type": "button", "bbox": (260, 250, 400, 290), "label": "Log in"},
    ]


def dashboard_elements(state):
    return [
        {"id": "card_profile",   "type": "button", "bbox": (40, 100, 200, 220),  "label": "Profile"},
        {"id": "card_settings",  "type": "button", "bbox": (240, 100, 400, 220), "label": "Settings"},
        {"id": "card_billing",   "type": "button", "bbox": (440, 100, 600, 220), "label": "Billing"},
        {"id": "logout_button",  "type": "button", "bbox": (480, 330, 600, 370), "label": "Log out"},
    ]


SCREENS = {"login": login_elements, "dashboard": dashboard_elements}


def initial_state():
    return {"screen": "login", "username": "", "password": "",
            "focus": None, "error": "", "opened_card": None}


def get_elements(state):
    return SCREENS[state["screen"]](state)


def render(state):
    # 观察函数 ω: state -> 像素。agent 只能看到这张图。
    img = Image.new("RGB", (W, H), (245, 246, 248))
    d = ImageDraw.Draw(img)
    d.text((20, 15), "ToyOS - " + state["screen"].upper(), fill=(20, 20, 20))
    for el in get_elements(state):
        x0, y0, x1, y1 = el["bbox"]
        if el["type"] == "input":
            focused = state["focus"] == el["id"]
            d.rectangle(el["bbox"], fill=(255, 255, 255),
                        outline=(0, 90, 200) if focused else (160, 160, 160), width=2)
            content = state.get(el["id"].replace("_field", ""), "")
            shown = "*" * len(content) if "password" in el["id"] else content
            d.text((x0 + 8, y0 + 10), shown if shown else el["label"],
                   fill=(30, 30, 30) if shown else (150, 150, 150))
        else:
            d.rectangle(el["bbox"], fill=(0, 110, 220), outline=(0, 70, 160), width=2)
            d.text((x0 + 12, y0 + 12), el["label"], fill=(255, 255, 255))
    if state["error"]:
        d.text((200, 300), state["error"], fill=(200, 30, 30))
    if state["opened_card"]:
        d.text((40, 250), "Opened card: " + state["opened_card"], fill=(0, 120, 0))
    return img


def _topmost_hit(x, y, elements):
    # 列表顺序即 z 序：后画的覆盖先画的，所以保留最后一个命中
    hit = None
    for el in elements:
        x0, y0, x1, y1 = el["bbox"]
        if x0 <= x <= x1 and y0 <= y <= y1:
            hit = el
    return hit


def apply_action(state, action):
    # 转移函数。返回 (new_state, info)；info 用于评测时归因。
    s = copy.deepcopy(state)
    s["error"] = ""
    info = {"hit": None, "event": "noop"}
    if not isinstance(action, dict) or "action" not in action:
        info["event"] = "invalid_action"
        return s, info
    if action["action"] == "click":
        el = _topmost_hit(action.get("x", -1), action.get("y", -1), get_elements(s))
        if el is None:
            s["focus"] = None
            info["event"] = "click_miss"
            return s, info
        info["hit"] = el["id"]
        if el["type"] == "input":
            s["focus"] = el["id"]
            info["event"] = "focus"
        elif el["id"] == "login_button":
            if s["username"] and s["password"]:
                s["screen"], s["focus"] = "dashboard", None
                info["event"] = "login_ok"
            else:
                s["error"] = "Missing credentials"
                info["event"] = "login_fail"
        elif el["id"].startswith("card_"):
            s["opened_card"] = el["id"].removeprefix("card_")
            info["event"] = "open_card"
        elif el["id"] == "logout_button":
            s = initial_state()
            info["event"] = "logout"
    elif action["action"] == "type":
        if s["focus"] in ("username_field", "password_field"):
            key = s["focus"].replace("_field", "")
            s[key] = s[key] + str(action.get("text", ""))
            info["event"] = "type"
        else:
            info["event"] = "type_no_focus"
    else:
        info["event"] = "unknown_action"
    return s, info


def task_success(state):
    # 结果态检查：登录成功且打开了 Settings 卡片
    return state["screen"] == "dashboard" and state["opened_card"] == "settings"


def oracle_action(state):
    # 作弊策略：直接读 ground-truth state，返回正确下一步（用于验证环境 + mock 回退）
    els = {e["id"]: e for e in get_elements(state)}

    def center(eid):
        x0, y0, x1, y1 = els[eid]["bbox"]
        return {"action": "click", "x": (x0 + x1) // 2, "y": (y0 + y1) // 2}

    if state["screen"] == "login":
        if state["focus"] != "username_field" and not state["username"]:
            return center("username_field")
        if state["focus"] == "username_field" and not state["username"]:
            return {"action": "type", "text": "alice"}
        if state["focus"] != "password_field" and not state["password"]:
            return center("password_field")
        if state["focus"] == "password_field" and not state["password"]:
            return {"action": "type", "text": "hunter2"}
        return center("login_button")
    return center("card_settings")


# oracle 演练：验证环境正确性
state = initial_state()
for t in range(8):
    if task_success(state):
        print(f"  step {t}: task done ✅")
        break
    a = oracle_action(state)
    state, info = apply_action(state, a)
    print(f"  step {t}: {json.dumps(a, ensure_ascii=False):<46} -> {info['event']}")
render(state)  # 最终截图：Dashboard + Opened card: settings

## 2. 策略层：VLM 双路径 + 动作解析

闭环里模型的位置：`render(state)` 的截图进去，**自由文本**出来，`parse_action` 把它抠成结构化动作。
解析失败本身就是一类要单独统计的错误（parse error ≠ grounding error ≠ decision error）。

- **VLM 路径**：`Qwen/Qwen2-VL-2B-Instruct`，标准 `apply_chat_template` + `qwen_vl_utils.process_vision_info` 流程，
  默认 bf16（约 5 GB 显存，Colab T4 够用，不需要量化）；显存特别紧张时可切 4-bit（约 2.5 GB）。**只看像素**，prompt 要求只输出一个 JSON 动作。
- **mock 路径**：`noisy_policy` 包装 oracle——以概率 `p_step` 返回正确动作，否则随机 misclick。
  它的作用是**把策略可靠性变成可控变量**，让我们能在第 4 节做受控的错误级联实验（真 VLM 的 p_step 不可控）。

In [ ]:
def parse_action(text):
    # 从自由文本中抠出第一个 JSON 对象并校验协议；失败返回 None
    m = re.search(r"\{.*?\}", text, re.S)
    if not m:
        return None
    try:
        a = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    if not isinstance(a, dict) or "action" not in a:
        return None
    return a


def noisy_policy(state, p_step=0.9, rng=None):
    # mock：以 p_step 概率给出 oracle 动作，否则随机位置 misclick
    rng = rng or random
    if rng.random() < p_step:
        return oracle_action(state)
    return {"action": "click", "x": rng.randrange(W), "y": rng.randrange(H)}


QWEN_ID = "Qwen/Qwen2-VL-2B-Instruct"
ACTION_PROMPT = (
    "You are a GUI agent operating a 640x400 screen. "
    "Task: log in with username 'alice' and password 'hunter2', then open the Settings card. "
    "Click an input field to focus it before typing. "
    "Look at the screenshot and output ONLY the single next action as one JSON object, e.g. "
    '{"action": "click", "x": 330, "y": 270} or {"action": "type", "text": "alice"}. '
    "Coordinates are pixels with origin at the top-left. No other text."
)


def load_qwen(use_4bit=False):
    # 资源：权重下载约 4.4 GB；显存 bf16 约 5 GB（默认，Colab T4 够用）/ 4-bit 约 2.5 GB（显存特别紧张时）
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
    kwargs = dict(device_map="auto")
    if use_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    else:
        kwargs["torch_dtype"] = "auto"
    model = Qwen2VLForConditionalGeneration.from_pretrained(QWEN_ID, **kwargs)
    processor = AutoProcessor.from_pretrained(QWEN_ID)
    return model, processor


def make_vlm_policy(use_4bit=False):
    from qwen_vl_utils import process_vision_info
    model, processor = load_qwen(use_4bit=use_4bit)

    def vlm_policy(state):
        messages = [{"role": "user", "content": [
            {"type": "image", "image": render(state)},
            {"type": "text", "text": ACTION_PROMPT},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                           padding=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        out = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
        raw = processor.batch_decode(out, skip_special_tokens=True)[0]
        action = parse_action(raw)
        # parse 失败 -> 退化为必然 miss 的点击（计为该步失败，而非崩溃）
        return action if action is not None else {"action": "click", "x": -1, "y": -1}

    return vlm_policy


vlm_policy = None
if USE_VLM:
    try:
        vlm_policy = make_vlm_policy()  # 首次运行下载约 4.4 GB
    except Exception as e:
        # 下载中断 / 显存不足 / 依赖版本不符等任何失败 -> 回退 mock，notebook 不崩
        print(f"VLM 加载失败 ({type(e).__name__}: {e}) -> 回退 mock 脚本化策略")
        USE_VLM = False
print("VLM policy:", "loaded" if vlm_policy else "skipped -> 用 mock 回退")

### 2.1 单步 smoke test：截图进、动作 JSON 出

> ⚠️ **资源标注**：若上一个 cell 里 `USE_VLM=True`，首次运行已触发 **约 4.4 GB** 权重下载；
> 推理显存 **bf16 约 5 GB（默认，Colab T4 够用）/ 4-bit 约 2.5 GB**（显存特别紧张时可选）。任何加载/推理失败都会 `try/except`
> 回退到 mock 脚本化策略（`noisy_policy` 包装 oracle），全 notebook 不会因此中断——只是把
> "真 VLM 看图" 换成 "脚本读 state"，评测链路完全一致。

跑完整 episode 之前先做最小闭环检查：`render(state)` 的 PIL 截图喂给策略 → 拿回一个动作 JSON →
`apply_action` 执行 → 看事件与新截图。调 GUI agent 的第一反射：**先确认单步闭环通，再谈成功率**。

In [ ]:
demo_state = initial_state()
if vlm_policy is not None:
    policy_name, demo_policy = "Qwen2-VL-2B", vlm_policy
else:
    policy_name, demo_policy = "mock(oracle)", lambda s: noisy_policy(s, p_step=1.0)

try:
    action = demo_policy(demo_state)  # PIL 截图进 -> 动作 JSON 出
except Exception as e:
    # 推理期失败（如 OOM）同样回退 mock，并让后续 episode 实验也走 mock
    print(f"VLM 推理失败 ({type(e).__name__}) -> 回退 mock")
    USE_VLM, vlm_policy = False, None
    policy_name, demo_policy = "mock(oracle)", lambda s: noisy_policy(s, p_step=1.0)
    action = demo_policy(demo_state)

print(f"[{policy_name}] 动作 JSON: {json.dumps(action, ensure_ascii=False)}")
demo_state, info = apply_action(demo_state, action)
print(f"环境事件: {info['event']}   命中元素: {info['hit']}")
render(demo_state)  # 新观察：若动作正确，Username 输入框应已聚焦（蓝色边框）

## 3. Set-of-marks：把坐标回归降级为选择题

**裸坐标协议**要求模型完成像素级回归——grounding 误差直接变成 miss（讲解第 3 节：$140\times40$ 按钮在
$\sigma=20$px 误差下命中率只剩 ~68%）。**set-of-marks（SoM）** 在渲染时给每个可交互元素叠加编号徽标，
协议升级为 `{"action": "click", "mark": N}`：模型只需读出编号，坐标由 `mark_map` 在 harness 侧解析。

下面做两个对比实验（无需 GPU，确定性可复现）：
1. **解析层**：几条典型的模型原始输出，看裸坐标协议的失败形态（夹杂解释文字 / 归一化坐标歧义 / 无 JSON）；
2. **grounding 层**：对落点误差 $\sigma$ 扫描，实测裸坐标命中率 vs 理论值
   $\operatorname{erf}\!\big(\tfrac{w}{2\sqrt2\sigma}\big)\operatorname{erf}\!\big(\tfrac{h}{2\sqrt2\sigma}\big)$，
   并对照 SoM（命中率 = 选对编号的概率，与像素误差**无关**）。

SoM 的代价：依赖元素检测器（这里我们有 ground-truth 元素表，真实系统没有），漏检的元素模型永远点不到。

In [ ]:
def render_with_marks(state):
    # SoM：在每个可交互元素左上角叠加编号徽标，返回 (图, mark->element_id 映射)
    img = render(state)
    d = ImageDraw.Draw(img)
    mark_map = {}
    for i, el in enumerate(get_elements(state), start=1):
        x0, y0 = el["bbox"][0], el["bbox"][1]
        d.rectangle((x0 - 1, y0 - 16, x0 + 17, y0), fill=(220, 40, 40))
        d.text((x0 + 4, y0 - 14), str(i), fill=(255, 255, 255))
        mark_map[i] = el["id"]
    return img, mark_map


def resolve_mark_action(action, state, mark_map):
    # 把 {"action":"click","mark":N} 解析回该元素中心的坐标点击
    if isinstance(action, dict) and action.get("action") == "click" and "mark" in action:
        eid = mark_map.get(action["mark"])
        for el in get_elements(state):
            if el["id"] == eid:
                x0, y0, x1, y1 = el["bbox"]
                return {"action": "click", "x": (x0 + x1) // 2, "y": (y0 + y1) // 2}
        return {"action": "click", "x": -1, "y": -1}  # 编号不存在 -> 必 miss
    return action


# --- 实验 1：解析层失败形态 ---
raw_outputs = [
    '{"action": "click", "x": 330, "y": 268}',            # 干净 JSON
    '好的，我现在点击登录按钮 {"action": "click", "x": 327, "y": 271}',  # 夹杂解释文字
    '{"action": "click", "x": 0.52, "y": 0.67}',          # 归一化坐标! 解析成功但必 miss
    'I will click the login button now.',                  # 没有 JSON -> parse error
    '{"action": "click", "mark": 3}',                      # SoM 协议
]
print("解析层：")
for raw in raw_outputs:
    print(f"  {raw[:44]!r:<50} -> {parse_action(raw)}")

# --- 实验 2：grounding 层，sigma 扫描 ---
def hit_rate_bare(sigma, n=4000, seed=0):
    # 目标 login_button: bbox (260,250,400,290)，瞄准中心 (330,270)
    rng = random.Random(seed)
    els = get_elements(initial_state())
    ok = 0
    for _ in range(n):
        hit = _topmost_hit(330 + rng.gauss(0, sigma), 270 + rng.gauss(0, sigma), els)
        ok += int(hit is not None and hit["id"] == "login_button")
    return ok / n


def hit_rate_theory(sigma, w=140, h=40):
    return math.erf(w / (2 * math.sqrt(2) * sigma)) * math.erf(h / (2 * math.sqrt(2) * sigma))


SOM_RATE = 0.97  # SoM: 命中率≈选对编号的概率（设 3% 选错），与像素误差无关
print("\ngrounding 层（目标 140x40 按钮）：")
print(f"  {'sigma/px':>8} | {'裸坐标 实测':>10} | {'裸坐标 理论':>10} | {'SoM':>5}")
for sigma in [2, 5, 10, 20, 40, 80]:
    print(f"  {sigma:>8} | {hit_rate_bare(sigma):>10.3f} | {hit_rate_theory(sigma):>10.3f} | {SOM_RATE:>5.2f}")

render_with_marks(initial_state())[0]  # 看一眼带编号徽标的截图

### 3.1 协议级对照：裸坐标 vs set-of-marks

把上面两层合到一张表里：构造一个**受控的输出分布**（各失败形态的混合比例是假设值，重点不是绝对数字，
而是分层归因的方法），对同一目标（login 屏的 `Log in` 按钮）模拟 N 次模型原始输出，统计：

- **解析成功率**：`parse_action` 能抠出合法动作的比例（协议解析层）；
- **端到端命中率**：动作最终落在目标元素上的比例（解析 + grounding 全链路）。

裸坐标协议的预期失败形态：夹杂解释文字（可救）、归一化坐标（解析"成功"但必 miss——最阴险的一类）、
无 JSON、像素落点误差 $\sigma$；SoM 协议没有像素误差，但会选错编号、也会输出无 JSON。
看点：**裸坐标的解析成功率并不低，掉链子掉在端到端**——这正是讲解第 2 节"解析错误会被误归因为能力差"的量化版本。

In [ ]:
def eval_protocol(protocol, n=3000, sigma=15, seed=1):
    # 受控对照：在 login 屏（已填好凭据）模拟"点击 Log in 按钮"这一步的模型原始输出
    rng = random.Random(seed)
    st = initial_state()
    st["username"], st["password"] = "alice", "hunter2"
    _, mark_map = render_with_marks(st)
    target_mark = {v: k for k, v in mark_map.items()}["login_button"]
    other_marks = [m for m in mark_map if m != target_mark]
    n_parsed = n_hit = 0
    for _ in range(n):
        r = rng.random()
        if protocol == "bare":
            x, y = 330 + rng.gauss(0, sigma), 270 + rng.gauss(0, sigma)
            if r < 0.70:    # 干净 JSON（仍带像素落点误差）
                raw = f'{{"action": "click", "x": {x:.0f}, "y": {y:.0f}}}'
            elif r < 0.85:  # 夹杂解释文字——parse_action 能救回来
                raw = f'好的，点击 Log in: {{"action": "click", "x": {x:.0f}, "y": {y:.0f}}}'
            elif r < 0.93:  # 归一化坐标——解析成功但必 miss
                raw = f'{{"action": "click", "x": {x / W:.3f}, "y": {y / H:.3f}}}'
            else:           # 无 JSON -> parse error
                raw = "I will click the login button now."
        else:  # set-of-marks
            if r < 0.90:    # 选对编号
                raw = f'{{"action": "click", "mark": {target_mark}}}'
            elif r < 0.95:  # 选错编号
                raw = f'{{"action": "click", "mark": {rng.choice(other_marks)}}}'
            else:           # 无 JSON -> parse error
                raw = f"Click mark {target_mark}."
        a = parse_action(raw)
        if a is None:
            continue
        n_parsed += 1
        a = resolve_mark_action(a, st, mark_map)
        hit = _topmost_hit(a.get("x", -1), a.get("y", -1), get_elements(st))
        n_hit += int(hit is not None and hit["id"] == "login_button")
    return n_parsed / n, n_hit / n


print(f"目标: login_button (140x40), 落点误差 sigma=15px, n=3000")
print(f"  {'协议':<14} | {'解析成功率':>8} | {'端到端命中率':>10}")
for label, proto in [("裸坐标", "bare"), ("set-of-marks", "som")]:
    pr, hr = eval_protocol(proto)
    print(f"  {label:<14} | {pr:>8.3f} | {hr:>10.3f}")
print("\n结论: SoM 把命中率与像素误差解耦(命中率=选对编号率)；"
      "裸坐标'解析成功'≠'命中'，归一化坐标歧义是静默杀手。")

## 4. Episode 评测：错误级联实测 vs 理论

评测协议：
- **步级成功**：该步动作与 oracle 动作在当前状态下产生相同的 `(event, hit)`——这比"坐标完全相等"更公平
  （点中按钮任意位置都算对）；
- **episode 成功**：`max_steps` 内 `task_success(state)` 为真（结果态检查）；
- **strict 成功**：episode 成功且全程无错步——对应错误级联公式的"错误不可恢复"假设。

先跑 **N=5** 个 episodes 看逐条轨迹（VLM 或 mock），再用 mock 大样本（500 episodes）对照理论 $S(n)=p^n$。
注意两个偏离方向（讲解第 4 节）：本 toy 环境里 misclick 几乎都**可恢复**（oracle/agent 看新状态能重试），
所以放宽步数上限后实测 > 理论；把 `max_steps` 卡死在 6（oracle 路径长度，无重试余地）时实测 ≈ 理论。

In [ ]:
def step_matches(state, action, intended):
    # 步级成功判据：两个动作在当前状态下产生相同的 (event, hit)
    _, ia = apply_action(state, action)
    _, ib = apply_action(state, intended)
    return (ia["event"], ia["hit"]) == (ib["event"], ib["hit"])


def run_episode(policy, max_steps=10, verbose=False):
    state = initial_state()
    correct, total, first_error = 0, 0, None
    for t in range(max_steps):
        if task_success(state):
            break
        intended = oracle_action(state)
        action = policy(state)
        ok = step_matches(state, action, intended)
        correct += int(ok)
        total += 1
        if not ok and first_error is None:
            first_error = t
        state, info = apply_action(state, action)
        if verbose:
            mark = "✓" if ok else "✗"
            print(f"    t={t} {json.dumps(action, ensure_ascii=False):<46} -> {info['event']:<14} {mark}")
    success = task_success(state)
    return {"success": success,
            "strict_success": success and first_error is None,
            "steps": total,
            "step_acc": correct / max(total, 1)}


# --- N=5 episodes：逐条轨迹 ---
N_EP = 5
rng = random.Random(0)
if USE_VLM:
    active_policy, name = vlm_policy, "Qwen2-VL-2B"
else:
    active_policy, name = (lambda s: noisy_policy(s, p_step=0.9, rng=rng)), "mock (p_step=0.9)"
print(f"=== {name}, {N_EP} episodes, max_steps=10 ===")
results = []
for ep in range(N_EP):
    print(f"  episode {ep}:")
    results.append(run_episode(active_policy, max_steps=10, verbose=True))
step_acc = sum(r["step_acc"] for r in results) / N_EP
ep_rate = sum(r["success"] for r in results) / N_EP
print(f"\n  步级成功率(均值) = {step_acc:.2f}   episode 成功率 = {ep_rate:.2f}  (N={N_EP}，方差很大!)")

# --- 大样本 mock：实测 vs 错误级联理论 ---
P_STEP, N_ORACLE, N_SIM = 0.95, 6, 500
theory = P_STEP ** N_ORACLE
rng2 = random.Random(42)
pol = lambda s: noisy_policy(s, p_step=P_STEP, rng=rng2)
no_retry = [run_episode(pol, max_steps=N_ORACLE) for _ in range(N_SIM)]
retry = [run_episode(pol, max_steps=12) for _ in range(N_SIM)]
print(f"\n=== 错误级联：p_step={P_STEP}, oracle 路径 n={N_ORACLE}, {N_SIM} episodes ===")
print(f"  理论 p^n                     = {theory:.3f}")
print(f"  实测 max_steps=6 (无重试余地) = {sum(r['success'] for r in no_retry)/N_SIM:.3f}")
print(f"  实测 max_steps=12 (允许重试)  = {sum(r['success'] for r in retry)/N_SIM:.3f}  <- 可恢复错误使实测高于理论")
print(f"  实测 strict (全程无错步)      = {sum(r['strict_success'] for r in retry)/N_SIM:.3f}  <- 回到 p^n 附近")

## ✏️ 练习 1：hit_test —— 命中检测原语

命中检测是 GUI harness 的最底层原语：环境每收到一次 `click` 都要回答"点到了谁"。
实现 `hit_test(x, y, elements)`：返回命中元素的 **id**（字符串）；无命中返回 `None`。

规则：
- bbox 为 `(x0, y0, x1, y1)`，边界**含端点**（`x0 <= x <= x1`）；
- 多个元素重叠时取**最上层**——列表顺序即 z 序，越靠后越上层（与 `render` 的绘制顺序一致）。

提示：参考本 notebook 的 `_topmost_hit`（它返回整个元素 dict，你返回 id），5 行以内可完成。

In [ ]:
def hit_test(x, y, elements):
    # TODO: 返回 (x, y) 命中的元素 id（str）；无命中返回 None。
    #       重叠时取列表中更靠后的元素（最上层）。
    raise NotImplementedError

In [ ]:
# --- 练习 1 自测：含重叠与边界用例 ---
ex1_elements = [
    {"id": "below", "type": "button", "bbox": (10, 10, 100, 100), "label": "B"},
    {"id": "above", "type": "button", "bbox": (50, 50, 150, 150), "label": "A"},
]
assert hit_test(20, 20, ex1_elements) == "below"          # 只在下层
assert hit_test(60, 60, ex1_elements) == "above"          # 重叠区 -> 最上层
assert hit_test(120, 120, ex1_elements) == "above"        # 只在上层
assert hit_test(200, 200, ex1_elements) is None           # 空白
assert hit_test(10, 10, ex1_elements) == "below"          # 边界含端点
assert hit_test(100, 100, ex1_elements) == "above"        # 角点同时在两个 bbox 内 -> 上层
assert hit_test(0, 0, []) is None                         # 空元素表
# 真实屏幕
login_els = get_elements(initial_state())
assert hit_test(330, 270, login_els) == "login_button"
assert hit_test(5, 5, login_els) is None
print("✅ 练习 1 通过")

## ✏️ 练习 2：错误级联理论值 vs 实测

实现 `episode_success_theory(p_step, n_steps)`：错误级联理想模型（每步独立、成功率恒为
`p_step`、错误不可恢复）下 $n$ 步任务的 episode 成功率 $S(n)$（讲解第 4 节）。

自测会把它与 mock 策略在"无重试余地"条件（`max_steps` = oracle 路径长度 6）下的实测值对比——
实测应落在理论值附近（toy 环境里随机 misclick 偶尔会碰巧点对，所以实测略高于理论是正常的）。
1 行可完成。

In [ ]:
def episode_success_theory(p_step, n_steps):
    # TODO: 返回错误级联理想模型下 n_steps 步任务的 episode 成功率 S(n)
    raise NotImplementedError

In [ ]:
# --- 练习 2 自测 ---
assert abs(episode_success_theory(0.95, 10) - 0.5987) < 1e-3   # 讲解第 4 节的数字
assert episode_success_theory(1.0, 50) == 1.0                  # 完美策略不衰减
assert abs(episode_success_theory(0.9, 1) - 0.9) < 1e-12       # 单步退化
assert episode_success_theory(0.5, 2) == 0.25
assert episode_success_theory(0.9, 20) < episode_success_theory(0.9, 5)  # 随步数单调下降

# 与实测对比：p_step=0.9，max_steps=6（oracle 路径长度，无重试余地）
ex2_rng = random.Random(7)
ex2_pol = lambda s: noisy_policy(s, p_step=0.9, rng=ex2_rng)
measured = sum(run_episode(ex2_pol, max_steps=6)["success"] for _ in range(400)) / 400
theory = episode_success_theory(0.9, 6)
print(f"理论 p^n = {theory:.3f}   实测(400 episodes) = {measured:.3f}")
assert abs(measured - theory) < 0.10, "实测与 p^n 偏差过大——检查实现"
print("✅ 练习 2 通过")

## ✏️ 练习 3：给 toy app 加第三屏（Settings 详情页）

体感"环境工程成本"（讲解第 8 节：每加一个任务/界面都要加元素表 + 转移 + 判分）。给状态机加
第三屏 `"settings"`，并实现扩展转移函数 `apply_action_v3`：

1. `settings_elements(state)`：返回 Settings 详情页元素表（列表顺序即 z 序），**恰好包含**：
   - `{"id": "toggle_dark", "type": "button", "bbox": (40, 100, 240, 150), "label": "Dark mode"}`
   - `{"id": "back_button", "type": "button", "bbox": (40, 330, 160, 370), "label": "Back"}`
2. `apply_action_v3(state, action)`：先调用原 `apply_action` 得 `(s, info)`，再按**入参 state 的屏幕**
   叠加三条新转移：
   - dashboard 上命中 `card_settings` → `s["screen"] = "settings"`（原有 `opened_card` 行为保留）；
   - settings 上命中 `back_button` → `s["screen"] = "dashboard"`；
   - settings 上命中 `toggle_dark` → 翻转 `s["dark_mode"]`（默认 `False`）。

提示：自测会先把 `settings_elements` 注册进 `SCREENS`，这样 `apply_action` 内部的命中检测
（`get_elements`）就能在第三屏上工作；`info["hit"]` 已经告诉你命中了谁。

In [ ]:
def settings_elements(state):
    # TODO: 返回 Settings 详情页的元素列表（见练习说明中的两个元素）
    raise NotImplementedError


def apply_action_v3(state, action):
    # TODO: s, info = apply_action(state, action)，再按入参 state["screen"] 与 info["hit"]
    #       叠加三条新转移（dashboard->settings / settings->dashboard / 翻转 dark_mode），
    #       返回 (s, info)
    raise NotImplementedError

In [ ]:
# --- 练习 3 自测：状态机转移正确性 ---
SCREENS["settings"] = settings_elements  # 注册第三屏，apply_action 的命中检测即可用

def _center_click(el):
    x0, y0, x1, y1 = el["bbox"]
    return {"action": "click", "x": (x0 + x1) // 2, "y": (y0 + y1) // 2}

s = initial_state()
s["screen"], s["username"], s["password"] = "dashboard", "alice", "hunter2"
set_els = {e["id"]: e for e in settings_elements(s)}
assert set(set_els) == {"toggle_dark", "back_button"}, "元素 id 不符"

# dashboard --card_settings--> settings（且保留 opened_card 语义）
dash_els = {e["id"]: e for e in get_elements(s)}
s2, info = apply_action_v3(s, _center_click(dash_els["card_settings"]))
assert s2["screen"] == "settings" and s2["opened_card"] == "settings"

# settings 上翻转 dark_mode：False -> True -> False
s3, _ = apply_action_v3(s2, _center_click(set_els["toggle_dark"]))
assert s3.get("dark_mode") is True
s3b, _ = apply_action_v3(s3, _center_click(set_els["toggle_dark"]))
assert s3b.get("dark_mode") is False

# settings --back--> dashboard
s4, _ = apply_action_v3(s3, _center_click(set_els["back_button"]))
assert s4["screen"] == "dashboard"

# 原有两屏的转移不受影响
s5, _ = apply_action_v3(initial_state(), {"action": "click", "x": 330, "y": 137})
assert s5["screen"] == "login" and s5["focus"] == "username_field"
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。每题参考实现与上面的自测 cell 兼容：运行参考 cell 后，回去重跑对应自测 cell 应全部通过。

In [ ]:
# 先自己做，再对照 —— 练习 1 参考实现
def hit_test(x, y, elements):
    hit = None
    for el in elements:  # 顺序扫描，后命中者覆盖前者 = 取最上层
        x0, y0, x1, y1 = el["bbox"]
        if x0 <= x <= x1 and y0 <= y <= y1:
            hit = el["id"]
    return hit

In [ ]:
# 先自己做，再对照 —— 练习 2 参考实现
def episode_success_theory(p_step, n_steps):
    return p_step ** n_steps

In [ ]:
# 先自己做，再对照 —— 练习 3 参考实现
def settings_elements(state):
    return [
        {"id": "toggle_dark", "type": "button", "bbox": (40, 100, 240, 150), "label": "Dark mode"},
        {"id": "back_button", "type": "button", "bbox": (40, 330, 160, 370), "label": "Back"},
    ]


def apply_action_v3(state, action):
    s, info = apply_action(state, action)  # 原有转移全部复用
    hit = info["hit"]
    if state["screen"] == "dashboard" and hit == "card_settings":
        s["screen"] = "settings"
        info["event"] = "open_settings"
    elif state["screen"] == "settings" and hit == "back_button":
        s["screen"] = "dashboard"
        info["event"] = "back"
    elif state["screen"] == "settings" and hit == "toggle_dark":
        s["dark_mode"] = not s.get("dark_mode", False)
        info["event"] = "toggle_dark"
    return s, info

## 小结

- **环境即评测资产**：`render` / `apply_action` / `task_success` 这个三件套就是 OSWorld、WebArena
  的最小同构——可复位、可判分（结果态检查）、可复现；练习 3 让你体感"加一屏 = 加元素表 + 转移 + 判分"
  的环境工程成本，这正是讲解第 5、8 节强调的"benchmark 最贵的资产是环境"。
- **错误要分层归因**：parse error（归一化坐标歧义、无 JSON）≠ grounding error（落点偏出 bbox）≠
  decision error。3.1 节的协议对照显示 set-of-marks 把命中率与像素误差解耦，代价是依赖元素检测器
  （呼应讲解第 2、3 节）。
- **错误级联是一等公民**：实测验证了 $S(n)=p^n$ 在"无重试余地"时成立、可恢复错误使实测向上偏离——
  这是讲解第 4 节两个偏离方向的实验版本，也是分数必须按步数分桶、报多 seed 的原因。

**下一站**：模块 06 把本章的 toy 判分升级为方法论——pass^k 可靠性指标、LLM-as-judge 校准、
METR time-horizon 测量 → [06 · Agentic 评测方法论](../06_agentic_evals/06_讲解.html)

---
## 🎯 真实数据胶囊题：真实网页 DOM 上的指令 grounding

computer-use 的关键是把自然语言指令 ground 到具体 UI 元素。抓一个真实维基百科页面的 DOM，提取所有链接，实现把 `点击关于 X 的链接` ground 到文本最匹配的真实链接。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

url="https://en.wikipedia.org/wiki/Large_language_model"
p=_f(url, "wiki_llm.html", headers={"User-Agent":"Mozilla/5.0 (course-demo)"})
html=open(p,encoding="utf-8",errors="ignore").read()
links=re.findall(r'<a [^>]*href="(/wiki/[^":#]+)"[^>]*>([^<]+)</a>', html)
links=[(h,t) for h,t in links if len(t)>2][:300]
print(f"真实页面提取到 {len(links)} 个链接, 例:", links[:2])

**练习**：实现 `ground(instruction, links)`：把指令 ground 到链接文本与指令**词重叠最多**的那个，返回 (href, text)。词重叠用小写分词集合交集大小。

In [ ]:
def ground(instruction, links):
    # TODO: 指令分词(小写)；对每个 link 算其文本词与指令词的交集大小；返回交集最大的 link
    raise NotImplementedError


In [ ]:
# 自测：含 'machine learning' 的指令应 ground 到机器学习相关链接
href, text = ground("click the link about machine learning", links)
assert "machine" in text.lower() or "learning" in text.lower() or "machine_learning" in href.lower()
# 无关指令也能返回某个链接(不报错)
h2,t2=ground("transformer architecture", links); assert h2 and t2
print(f"grounding ✓  'machine learning' -> {text!r} ({href})")


### 📖 参考答案

In [ ]:
def ground(instruction, links):
    iw=set(re.findall(r"[a-z]+", instruction.lower()))
    def overlap(t): return len(iw & set(re.findall(r"[a-z]+", t.lower())))
    return max(links, key=lambda l: overlap(l[1]))
print("✓ grounding = 把指令映射到真实可操作元素，词重叠是最朴素的 baseline")